# Deteksi Sampah Organik vs Anorganik — YOLO26

- **Dataset:** [waste Detection](https://universe.roboflow.com/capstone-catzw/waste-detection-yle4x) (Roboflow Universe, CC BY 4.0) — 2 kelas `organic`/`inorganic`, sudah ber-bounding box
- **Model:** YOLO26n (Ultralytics) — versi terbaru, NMS-free, ringan; cadangan YOLO11n

## Konfigurasi

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd()

# Dataset Roboflow
RF_WORKSPACE = "capstone-catzw"
RF_PROJECT   = "waste-detection-yle4x"
RF_VERSION   = 1
RF_FORMAT    = "yolov8"

# Path (relatif terhadap folder workspace)
DATA_DIR    = Path("data")
RUNS_DIR    = Path("runs")
MODELS_DIR  = Path("models")
DATASET_DIR = DATA_DIR / f"{RF_PROJECT}-v{RF_VERSION}"

# Model
MODEL_NAME     = "yolo26n.pt"
FALLBACK_MODEL = "yolo11n.pt"
RUN_NAME       = "sampah_" + Path(MODEL_NAME).stem

# Hyperparameter
EPOCHS     = 50
IMG_SIZE   = 640
BATCH_SIZE = 16
PATIENCE   = 15
SEED       = 42

# Inference
CONF_THRESHOLD  = 0.40
N_SAMPLE_IMAGES = 6
GAMBAR_SENDIRI  = None    # mis. "foto_sampah.jpg"
RUN_WEBCAM      = False   # True hanya di laptop
WEBCAM_INDEX    = 0

LABEL_MAP = {"organic": "organik", "inorganic": "anorganik"}


def ambil_api_key(nama="ROBOFLOW_API_KEY"):
    if os.environ.get(nama):
        return os.environ[nama]
    env_file = ROOT / ".env"
    if env_file.exists():
        for baris in env_file.read_text().splitlines():
            if baris.strip().startswith(f"{nama}="):
                nilai = baris.split("=", 1)[1].strip().strip('"').strip("'")
                if nilai and nilai != "isi_api_key_di_sini":
                    return nilai
    from getpass import getpass
    return getpass(f"{nama}: ")


ROBOFLOW_API_KEY = None if (DATASET_DIR / "data.yaml").exists() else ambil_api_key()

for d in (DATA_DIR, RUNS_DIR, MODELS_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("Folder proyek:", ROOT)

## 1. Setup

In [ ]:
print("Python environment:", sys.executable)
print("Kalau dependency belum ada, jalankan `bash setup.sh` di terminal workspace.")

In [ ]:
import random
import shutil

import cv2
import matplotlib.pyplot as plt
import pandas as pd
import torch
import ultralytics
import yaml
from ultralytics import YOLO

random.seed(SEED)

print("Python     :", sys.version.split()[0])
print("PyTorch    :", torch.__version__)
print("Ultralytics:", ultralytics.__version__)
print("OpenCV     :", cv2.__version__)

In [ ]:
if torch.cuda.is_available():
    DEVICE = 0
    print("GPU:", torch.cuda.get_device_name(0))
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    DEVICE = "mps"
    print("GPU: Apple MPS")
else:
    DEVICE = "cpu"
    print("GPU tidak terdeteksi, memakai CPU")

In [ ]:
def tampilkan(img_bgr, judul=None, lebar_maks=640):
    if judul:
        print(judul)
    tinggi, lebar = img_bgr.shape[:2]
    if lebar > lebar_maks:
        img_bgr = cv2.resize(img_bgr, (lebar_maks, int(tinggi * lebar_maks / lebar)))
    plt.figure(figsize=(6, 6))
    plt.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.show()

## 2. Dataset

### Unduh dari Roboflow

In [ ]:
if (DATASET_DIR / "data.yaml").exists():
    print("Dataset sudah ada:", DATASET_DIR)
else:
    from roboflow import Roboflow

    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    project = rf.workspace(RF_WORKSPACE).project(RF_PROJECT)
    project.version(RF_VERSION).download(RF_FORMAT, location=str(DATASET_DIR))
    print("Dataset tersimpan:", DATASET_DIR)

### Rapikan data.yaml

In [ ]:
DATA_YAML = DATASET_DIR / "data.yaml"

with open(DATA_YAML) as f:
    data_cfg = yaml.safe_load(f)

data_cfg["path"]  = str(DATASET_DIR.resolve())
data_cfg["train"] = "train/images"
data_cfg["val"]   = "valid/images"
if (DATASET_DIR / "test" / "images").exists():
    data_cfg["test"] = "test/images"
else:
    data_cfg.pop("test", None)

with open(DATA_YAML, "w") as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False)

names = data_cfg["names"]
CLASS_NAMES = dict(enumerate(names)) if isinstance(names, list) else {int(k): v for k, v in names.items()}
print("Kelas:", CLASS_NAMES)

for nama in CLASS_NAMES.values():
    if nama not in LABEL_MAP:
        print(f"⚠️ '{nama}' belum ada di LABEL_MAP")

### Cek isi dataset

In [ ]:
SPLITS = [s for s in ["train", "valid", "test"] if (DATASET_DIR / s / "images").exists()]
EKSTENSI_GAMBAR = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def daftar_gambar(split):
    folder = DATASET_DIR / split / "images"
    return sorted(p for p in folder.iterdir() if p.suffix.lower() in EKSTENSI_GAMBAR)


def path_label(path_gambar):
    return path_gambar.parent.parent / "labels" / (path_gambar.stem + ".txt")


pd.DataFrame([
    {
        "split": s,
        "gambar": len(daftar_gambar(s)),
        "punya label": sum(path_label(p).exists() for p in daftar_gambar(s)),
    }
    for s in SPLITS
])

### Distribusi kelas

In [ ]:
def hitung_kelas(split):
    jumlah = {nama: 0 for nama in CLASS_NAMES.values()}
    for p in daftar_gambar(split):
        lbl = path_label(p)
        if lbl.exists():
            for baris in lbl.read_text().splitlines():
                if baris.strip():
                    jumlah[CLASS_NAMES[int(float(baris.split()[0]))]] += 1
    return jumlah


distribusi = pd.DataFrame({s: hitung_kelas(s) for s in SPLITS})
distribusi

In [ ]:
distribusi.plot(kind="bar", figsize=(6, 4))
plt.title("Jumlah objek per kelas")
plt.ylabel("jumlah kotak")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 3. Eksplorasi Data

In [ ]:
def baca_label(path_gambar):
    lbl = path_label(path_gambar)
    if not lbl.exists():
        return []
    hasil = []
    for baris in lbl.read_text().splitlines():
        angka = baris.split()
        if len(angka) >= 5:
            hasil.append((int(float(angka[0])), *map(float, angka[1:5])))
    return hasil

In [ ]:
WARNA = {"organik": (60, 180, 75), "anorganik": (230, 120, 30)}


def gambar_kotak(img, x1, y1, x2, y2, teks, warna):
    cv2.rectangle(img, (x1, y1), (x2, y2), warna, 2)
    (tw, th), _ = cv2.getTextSize(teks, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
    y_teks = max(y1, th + 6)
    cv2.rectangle(img, (x1, y_teks - th - 6), (x1 + tw + 4, y_teks), warna, -1)
    cv2.putText(img, teks, (x1 + 2, y_teks - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)


def gambar_label_asli(path_gambar):
    img = cv2.imread(str(path_gambar))
    H, W = img.shape[:2]
    for kelas_id, xc, yc, w, h in baca_label(path_gambar):
        x1, y1 = int((xc - w / 2) * W), int((yc - h / 2) * H)
        x2, y2 = int((xc + w / 2) * W), int((yc + h / 2) * H)
        nama = LABEL_MAP.get(CLASS_NAMES[kelas_id], CLASS_NAMES[kelas_id])
        gambar_kotak(img, x1, y1, x2, y2, nama, WARNA.get(nama, (0, 0, 255)))
    return img

In [ ]:
train_imgs = daftar_gambar("train")
for p in random.sample(train_imgs, k=min(N_SAMPLE_IMAGES, len(train_imgs))):
    tampilkan(gambar_label_asli(p), judul=p.name)

## 4. Training

In [ ]:
def muat_model(nama):
    return YOLO(str(MODELS_DIR / nama))


try:
    model = muat_model(MODEL_NAME)
    MODEL_DIPAKAI = MODEL_NAME
except Exception as e:
    print(f"Gagal memuat {MODEL_NAME}: {e} -> pakai {FALLBACK_MODEL}")
    model = muat_model(FALLBACK_MODEL)
    MODEL_DIPAKAI = FALLBACK_MODEL

print("Model:", MODEL_DIPAKAI)

In [ ]:
model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    patience=PATIENCE,
    seed=SEED,
    device=DEVICE,
    project=str(RUNS_DIR.resolve()),
    name=RUN_NAME,
    exist_ok=True,
)

In [ ]:
TRAIN_DIR    = Path(model.trainer.save_dir)
BEST_WEIGHTS = TRAIN_DIR / "weights" / "best.pt"
print("Hasil training:", TRAIN_DIR)
print("best.pt ada   :", BEST_WEIGHTS.exists())

## 5. Evaluasi

In [ ]:
best_model = YOLO(str(BEST_WEIGHTS))
metrics = best_model.val(
    data=str(DATA_YAML),
    split="val",
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    project=str(RUNS_DIR.resolve()),
    name=RUN_NAME + "_val",
    exist_ok=True,
)

### Metrik

In [ ]:
print(f"Precision : {metrics.box.mp:.3f}")
print(f"Recall    : {metrics.box.mr:.3f}")
print(f"mAP50     : {metrics.box.map50:.3f}")
print(f"mAP50-95  : {metrics.box.map:.3f}")

pd.DataFrame([
    {
        "kelas": LABEL_MAP.get(CLASS_NAMES[int(k)], CLASS_NAMES[int(k)]),
        "precision": round(float(metrics.box.p[i]), 3),
        "recall": round(float(metrics.box.r[i]), 3),
        "mAP50": round(float(metrics.box.ap50[i]), 3),
        "mAP50-95": round(float(metrics.box.ap[i]), 3),
    }
    for i, k in enumerate(metrics.ap_class_index)
])

### Grafik loss

In [ ]:
hasil_csv = pd.read_csv(TRAIN_DIR / "results.csv")
hasil_csv.columns = hasil_csv.columns.str.strip()

jenis_loss = sorted({c.split("/")[1] for c in hasil_csv.columns if "loss" in c})
fig, axes = plt.subplots(1, len(jenis_loss), figsize=(5 * len(jenis_loss), 4), squeeze=False)
for ax, jenis in zip(axes[0], jenis_loss):
    for bagian in ["train", "val"]:
        kolom = f"{bagian}/{jenis}"
        if kolom in hasil_csv:
            ax.plot(hasil_csv["epoch"], hasil_csv[kolom], label=bagian)
    ax.set_title(jenis)
    ax.set_xlabel("epoch")
    ax.legend()
plt.tight_layout()
plt.show()

### Grafik metrik

In [ ]:
kolom_metrik = {
    "metrics/precision(B)": "precision",
    "metrics/recall(B)": "recall",
    "metrics/mAP50(B)": "mAP50",
    "metrics/mAP50-95(B)": "mAP50-95",
}
plt.figure(figsize=(6, 4))
for kolom, label in kolom_metrik.items():
    if kolom in hasil_csv:
        plt.plot(hasil_csv["epoch"], hasil_csv[kolom], label=label)
plt.xlabel("epoch")
plt.ylim(0, 1)
plt.legend()
plt.tight_layout()
plt.show()

### Confusion matrix

In [ ]:
cm_path = TRAIN_DIR / "confusion_matrix_normalized.png"
if cm_path.exists():
    tampilkan(cv2.imread(str(cm_path)), lebar_maks=700)
else:
    print("Tidak ditemukan:", cm_path)

## 6. Inference

### Aturan output

In [ ]:
URUTAN_OUTPUT = ["organik", "anorganik"]


def aturan_output(kelas_terdeteksi):
    jenis = {LABEL_MAP[k] for k in kelas_terdeteksi if k in LABEL_MAP}
    return [o for o in URUTAN_OUTPUT if o in jenis]


def cetak_output(outputs):
    for o in outputs:
        print(o)


assert aturan_output([]) == []
assert aturan_output(["organic"]) == ["organik"]
assert aturan_output(["inorganic", "inorganic"]) == ["anorganik"]
assert aturan_output(["inorganic", "organic"]) == ["organik", "anorganik"]

### Fungsi deteksi

In [ ]:
def deteksi_sampah(sumber):
    hasil = best_model.predict(
        sumber, conf=CONF_THRESHOLD, imgsz=IMG_SIZE, device=DEVICE, verbose=False
    )[0]
    img = hasil.orig_img.copy()
    kelas_terdeteksi = []
    for kotak in hasil.boxes:
        x1, y1, x2, y2 = map(int, kotak.xyxy[0].tolist())
        nama = hasil.names[int(kotak.cls)]
        kelas_terdeteksi.append(nama)
        label = LABEL_MAP.get(nama, nama)
        gambar_kotak(img, x1, y1, x2, y2, f"{label} {float(kotak.conf):.2f}", WARNA.get(label, (0, 0, 255)))
    return aturan_output(kelas_terdeteksi), img

### Gambar uji

In [ ]:
split_uji = "test" if "test" in SPLITS else "valid"
gambar_uji = daftar_gambar(split_uji)

for p in random.sample(gambar_uji, k=min(N_SAMPLE_IMAGES, len(gambar_uji))):
    outputs, img_hasil = deteksi_sampah(str(p))
    tampilkan(img_hasil, judul=f"----- {p.name} -----")
    cetak_output(outputs)

### Gambar sendiri

In [ ]:
daftar = [GAMBAR_SENDIRI] if GAMBAR_SENDIRI else []
if not daftar:
    print("Lewati deteksi gambar tunggal; isi GAMBAR_SENDIRI dengan path gambar lokal untuk menjalankannya.")

for f in daftar:
    outputs, img_hasil = deteksi_sampah(f)
    tampilkan(img_hasil, judul=f"----- {f} -----")
    cetak_output(outputs)

### Webcam

In [ ]:
if RUN_WEBCAM:
    cap = cv2.VideoCapture(WEBCAM_INDEX)
    output_terakhir = None
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        outputs, img_hasil = deteksi_sampah(frame)
        if outputs != output_terakhir:
            cetak_output(outputs)
            output_terakhir = outputs
        cv2.imshow("Deteksi Sampah (Q untuk keluar)", img_hasil)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
    cap.release()
    cv2.destroyAllWindows()
else:
    print("Webcam mati (RUN_WEBCAM=False)")

## 7. Simpan Model

In [ ]:
MODEL_FINAL = MODELS_DIR / f"sampah_{Path(MODEL_DIPAKAI).stem}_best.pt"
shutil.copy2(BEST_WEIGHTS, MODEL_FINAL)
print("Model final:", MODEL_FINAL)

### Muat ulang

In [ ]:
model_tersimpan = YOLO(str(MODEL_FINAL))
print("Kelas:", model_tersimpan.names)